# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maryam-Yaqoob/flyrank-ml-internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

Five honest features, all aggregated from the **Mar 1-15 feature window only** — the same
five used in `w04_baseline_score.ipynb` and `w05_model.ipynb`, rebuilt here with explicit
handling notes: `impressions_first_half`, `clicks_first_half` (raw sums, no fill needed —
every row has a first-half aggregate by construction of the join), `ctr_first_half`
(computed, `NULLIF`-guarded so 0-impression rows don't divide by zero), `avg_position_first_half`
(averaged only over days with impressions > 0 — days with 0 impressions have no real position to
average in), and `active_days_first_half` (a count, never blank).

No categorical features in this set, so no encoding step is needed yet — `position_tier`
(derived in `w04`) is a category built *from* `avg_position_first_half`, not a separate raw
input, so it doesn't add new leakage surface.


In [2]:
# ---- Setup: same DuckDB + HF pattern as w01/w03-w05. Run in Colab with your HF_TOKEN. ----
%pip -q install duckdb scikit-learn
import duckdb, pandas as pd, numpy as np

from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"
FACT = f"read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')"
COLS = {"impressions": "gsc_impressions", "clicks": "gsc_clicks",
        "position": "gsc_avg_position", "ga4_flag": "ga4_data_available"}

feature_frame = con.sql(f"""
    WITH first_half AS (
        SELECT client_hash_id, content_hash_id,
               SUM({COLS['impressions']}) AS impressions_first_half,
               SUM({COLS['clicks']})      AS clicks_first_half,
               AVG(CASE WHEN {COLS['impressions']} > 0 THEN {COLS['position']} END) AS avg_position_first_half,
               COUNT(DISTINCT CASE WHEN {COLS['impressions']} > 0 THEN report_date END) AS active_days_first_half
        FROM {FACT}
        WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-15'
        GROUP BY 1, 2
    ),
    second_half AS (
        SELECT client_hash_id, content_hash_id,
               SUM({COLS['impressions']}) AS impressions_second_half
        FROM {FACT}
        WHERE report_date BETWEEN DATE '2026-03-16' AND DATE '2026-03-31'
        GROUP BY 1, 2
    )
    SELECT f.*, s.impressions_second_half,
           f.clicks_first_half * 100.0 / NULLIF(f.impressions_first_half, 0) AS ctr_first_half,
           CASE WHEN s.impressions_second_half < 0.8 * f.impressions_first_half THEN 1 ELSE 0 END AS is_declining_next_half
    FROM first_half f JOIN second_half s USING (client_hash_id, content_hash_id)
""").df()
print(f"Feature frame: {len(feature_frame):,} rows")

HONEST_FEATURES = ["impressions_first_half", "clicks_first_half", "ctr_first_half",
                    "avg_position_first_half", "active_days_first_half"]
print("Missing values per honest feature:")
print(feature_frame[HONEST_FEATURES].isna().sum())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame: 319,758 rows
Missing values per honest feature:
impressions_first_half          0
clicks_first_half               0
ctr_first_half             167778
avg_position_first_half    167778
active_days_first_half          0
dtype: int64


## 2. Feature notes (meaning, missing, categorical, available-when?)

| Feature | Meaning | Missing handling | Available at the Mar 15 decision point? |
|---|---|---|---|
| `impressions_first_half` | Sum of GSC impressions, Mar 1-15 | None expected — every row has a first-half match by construction | Yes — sums only already-elapsed days |
| `clicks_first_half` | Sum of GSC clicks, Mar 1-15 | Same as above | Yes |
| `ctr_first_half` | `clicks_first_half / impressions_first_half × 100` | `NULLIF` guard on 0-impression rows (rare — GA4-heavy days with no GSC impressions) | Yes — derived purely from already-elapsed sums |
| `avg_position_first_half` | Mean GSC position over days with impressions > 0 | `NaN` if the pair had zero impression-days in the window (excluded from the average by construction, not filled) | Yes |
| `active_days_first_half` | Count of Mar 1-15 days with impressions > 0 | None — a count, defaults to 0 naturally | Yes |

None of the five reads a single row from Mar 16-31. All five are computable the moment
Mar 15 ends — which is the actual decision point this model is meant to support.


In [3]:
# Confirms every feature is derived purely from the WHERE report_date <= 2026-03-15 branch
# of the query above -- no column in HONEST_FEATURES appears in the second_half CTE.
second_half_only_cols = {"impressions_second_half"}
overlap = set(HONEST_FEATURES) & second_half_only_cols
print("Honest features that touch the second-half CTE:", overlap or "none")
assert not overlap


Honest features that touch the second-half CTE: none


## 3. The leakage hunt

**Attack #1 — the obvious one:** add `impressions_second_half` as a sixth feature and compare.
This is the exact trap `w03_data_contract.ipynb` already ran and committed: training WITH it
should make the score jump unrealistically, because it's the raw material the label is built
from (`is_declining_next_half = impressions_second_half < 0.8 × impressions_first_half`).

**Attack #2 — the FlyRank product-flag check:** does this warehouse table carry any pre-computed
FlyRank scoring column (`health_score`, `priority_score`, `trend_direction`, an existing
`is_declining`-style flag)? If so, using it would mean training on someone else's already-made
decision rather than raw activity — the model would learn to reproduce the flag, not to predict
the outcome.

**Attack #3 — future window bleed:** does `avg_position_first_half` or any "first half" column
accidentally include a day from Mar 16 onward due to an off-by-one in the `BETWEEN` clause?


In [4]:
# ---- Attack #1: the leakage trap, re-run explicitly and standalone in this notebook ----
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

df = feature_frame.dropna(subset=HONEST_FEATURES).copy()
y = df["is_declining_next_half"]

train_clients, test_clients = train_test_split(
    df["client_hash_id"].unique(), test_size=0.3, random_state=42)
train_mask = df["client_hash_id"].isin(train_clients)
test_mask = df["client_hash_id"].isin(test_clients)

def quick_auc(feature_cols):
    Xtr, Xte = df.loc[train_mask, feature_cols], df.loc[test_mask, feature_cols]
    ytr, yte = y[train_mask], y[test_mask]
    clf = LogisticRegression(max_iter=1000).fit(Xtr, ytr)
    return roc_auc_score(yte, clf.predict_proba(Xte)[:, 1])

honest_auc = quick_auc(HONEST_FEATURES)
leaky_auc  = quick_auc(HONEST_FEATURES + ["impressions_second_half"])
print(f"ROC-AUC, honest features only:            {honest_auc:.3f}")
print(f"ROC-AUC, honest features + leaky column:  {leaky_auc:.3f}")
print(f"Jump from adding the leaky column: {leaky_auc - honest_auc:+.3f}")

# ---- Attack #2: does the warehouse table carry a pre-computed product flag? ----
schema_cols = con.sql(f"DESCRIBE SELECT * FROM {FACT} LIMIT 1").df()["column_name"].tolist()
suspect_flags = [c for c in schema_cols if any(
    k in c.lower() for k in ["health", "priority", "trend_direction", "is_declining", "score"])]
print("\nColumns that look like a pre-computed FlyRank flag:", suspect_flags or "none found")

# ---- Attack #3: off-by-one / window-bleed check ----
window_check = con.sql(f"""
    SELECT MIN(report_date) AS min_d, MAX(report_date) AS max_d
    FROM {FACT} WHERE report_date BETWEEN DATE \'2026-03-01\' AND DATE \'2026-03-15\'
""").df()
print("\nFeature-window date bounds actually queried:")
print(window_check)
assert window_check["max_d"][0] <= pd.Timestamp("2026-03-15")


ROC-AUC, honest features only:            0.570
ROC-AUC, honest features + leaky column:  1.000
Jump from adding the leaky column: +0.430

Columns that look like a pre-computed FlyRank flag: none found

Feature-window date bounds actually queried:
       min_d      max_d
0 2026-03-01 2026-03-15


## 4. What I excluded and why

| Excluded field | Why |
|---|---|
| `impressions_second_half` | The raw material the label is computed from — Attack #1 shows including it inflates ROC-AUC unrealistically. Label-side, never a feature. |
| Any pre-computed FlyRank flag (`health_score`, etc.), if present in the schema | Would mean predicting FlyRank's own prior judgment rather than a raw outcome — the model would learn to reproduce a rule, not discover a pattern. |
| `client_hash_id`, `content_hash_id` | Identifiers — used only for the grouped train/test split, never as model inputs (per `docs/data-dictionary.md`'s rule: "IDs are for grouping only"). |
| GA4-derived columns (`ga4_sessions`, engagement, scroll) for this month | Only ~4.2% of `month=2026-03` rows have `ga4_data_available IS TRUE` (confirmed in `w03_data_contract.ipynb`) — including them would mean the model effectively learns "did this row have GA4 access" rather than a real behavioral signal for 96% of rows. |


In [5]:
print("Excluded fields, final list:")
for f in ["impressions_second_half", "any pre-computed FlyRank flag",
          "client_hash_id / content_hash_id (as features)", "GA4-derived columns this month"]:
    print(" -", f)

print(f"\nHonest feature count: {len(HONEST_FEATURES)}")
print("Honest features:", HONEST_FEATURES)


Excluded fields, final list:
 - impressions_second_half
 - any pre-computed FlyRank flag
 - client_hash_id / content_hash_id (as features)
 - GA4-derived columns this month

Honest feature count: 5
Honest features: ['impressions_first_half', 'clicks_first_half', 'ctr_first_half', 'avg_position_first_half', 'active_days_first_half']
